In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time
import numpy as np
from scipy.spatial.transform import Rotation

import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation as jaxRotation

import matplotlib.pyplot as plt

from helicopter.vision.tracking import TrianglePointMatcher, ICP

In [ ]:
# measured_points = np.array([[      1.768,  -0.0079125,    -0.43461],
#        [     1.7783,    0.021053,    -0.44639],
#        [     1.7688,  0.00070465,    -0.44736],
#        [     1.7945,     0.11468,    -0.44259],
#        [     1.7672,   -0.043063,    -0.43921],
#        [     1.7704,   -0.026978,    -0.43964],
#        [     1.7892,     0.12382,    -0.45662]])

In [ ]:
table_space_measured = np.array([[   0.025213,    -0.49795,    -0.10622],
       [ -0.0037647,    -0.49748,   -0.090968],
       [   0.011092,    -0.50011,    -0.10072],
       [  -0.020689,    -0.49644,   -0.086575],
       [  -0.088161,    -0.43811,   -0.063245],
       [   -0.11827,    -0.42931,    -0.08105],
       [  -0.037543,    -0.48335,    -0.11287],
       [   -0.01458,    -0.49513,    -0.09577],
       [ 0.00086675,    -0.48929,   -0.076964]])

In [ ]:
reference_points = np.load('/home/ray/projects/helicopter/assets/point_clouds/green_syma.npy')

In [ ]:
def plot_3d(points, window, marker_size=15, c=None, show_indices=True, center=False, other_points=None):
    plt.close('all')

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection='3d')

    if center:
        centered_points = points - np.mean(points, axis=0)
        x = centered_points[:, 0]
        y = centered_points[:, 1]
        z = centered_points[:, 2]
    else:
        x = points[:, 0]
        y = points[:, 1]
        z = points[:, 2]

    if c is None:
        c = y

    ax.scatter(x, y, z, c=c, cmap='plasma', s=marker_size, depthshade=True)

    if show_indices:
        for _i in range(len(points)):
            ax.text(x[_i], y[_i], z[_i], str(_i), size=8, color='black', zorder=10)

    if other_points is not None:
        x = other_points[:, 0]
        y = other_points[:, 1]
        z = other_points[:, 2]

        ax.scatter(x, y, z, c='g', s=marker_size, depthshade=True)

    ax.set_xlim(-window, window)
    ax.set_ylim(-window, window)
    ax.set_zlim(-window, window)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    plt.show()

In [ ]:
matcher = TrianglePointMatcher(n=5000, k=100, inlier_threshold=0.01)
r, t = matcher.get_alignment(sample_points=table_space_measured)

In [ ]:
r.as_rotvec(degrees=True)

In [ ]:
t

In [ ]:
%matplotlib ipympl

plot_3d(r.apply(reference_points), other_points=(table_space_measured - t), window=0.1)

In [ ]:
icp = ICP(distance_threshold=1e-1, etol=0.003, max_iterations=10)

In [ ]:
def pad_points(points):
    max_size = len(reference_points)
    current_size = points.shape[0]

    pad_length = max_size - current_size

    pad_array = np.zeros((pad_length, 3), dtype=points.dtype)
    padded_points = np.vstack([points, pad_array])

    valid_input_mask = np.zeros(max_size, dtype=bool)
    valid_input_mask[:current_size] = True

    return padded_points, valid_input_mask

In [ ]:
q_jax = jaxRotation.from_quat(jnp.array(r.as_quat(canonical=True)))
padded, valid_mask = pad_points(table_space_measured)
start = time.perf_counter()
_, _, q_new, t_new = icp.iterate(q_old=q_jax, t_old=t, sample_points=padded, reference_points=reference_points, valid_input_mask=valid_mask)
end = time.perf_counter()

In [ ]:
end - start

In [ ]:
q_np = Rotation.from_quat(np.array(q_jax.as_quat(canonical=True)))

In [ ]:
q_np.as_rotvec(degrees=True)

In [ ]:
np.asarray(t_new)

In [ ]:
%matplotlib ipympl
plot_3d(q_new.apply(reference_points), other_points=(table_space_measured - np.asarray(t_new)), window=0.1)

In [ ]:
%matplotlib ipympl
plot_3d(table_space_measured, center=True, window=0.1)